# Dijet pointing-resolution systematic
Estimate the one-sided pointing uncertainty `abs(Reco/Ref - 1)` from `hRecoDijetPtEtaCMRefEtaCM_<eta cut>`. X is reconstructed pTave, Y is reco etaCM, and Z is matched-reference etaCM. Full shapes are unit-normalized; F/B is folded from unnormalized projections and always uses independent errors.

In [11]:
%load_ext autoreload
%autoreload 2
from dataclasses import replace
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / 'CMakeLists.txt').is_file()
        and (candidate / 'hist_analysis').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL,
    STANDARD_DIJET_ETA_CUT_INDEX,
    TEST_DIJET_PTAVE_BINS,
)
from hist_analysis.python.data_distributions import forward_backward_from_full
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import (
    normalize_histogram, ratio_to_nominal,
)
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.projections import project_histogram
from hist_analysis.python.root_style import (
    COLORS, DEFAULT_PLOT_STYLE, save_canvas,
)
from hist_analysis.python.systematic_fits import (
    calculate_one_sided_systematic,
    fit_histogram_variations,
    format_fit_summary_lines,
    smooth_systematic_running_max,
    write_systematic_csv,
)

ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration
Comparison ratios allow standard (`''`) or binomial (`'B'`) errors. `FORWARD_BACKWARD_RATIO_OPTION` is protected: F/B construction is never binomial.

In [ ]:
GENERATOR = 'embedding'  # embedding or pythia
DIRECTION = 'combined'   # combined, Pbgoing, or pgoing
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.4, 3.0)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BINS = tuple(TEST_DIJET_PTAVE_BINS)
REBIN_ETA = 2

SYSTEMATIC_EXTRACTION = 'fit'  # fit or bin_by_bin
APPLY_SYSTEMATIC_SMOOTHING = True
FULL_SMOOTHING_ORIGIN = -0.465 + 0.00001

FORWARD_BACKWARD_RATIO_OPTION = ''  # protected: never binomial
FULL_COMPARISON_RATIO_OPTION = 'B'  # '' or 'B'
FB_COMPARISON_RATIO_OPTION = 'B'    # '' or 'B'; after F/B construction

FULL_FIT_FUNCTION = 'pol4'
FB_FIT_FUNCTION = 'pol1'
FULL_FIT_INITIAL_VALUES = {
    'Reco / Ref': (1.0, 0.0, 0.0, 0.0, 0.0),
}
FB_FIT_INITIAL_VALUES = {
    '(F/B) Reco / Ref': (1.0, 0.0),
}
FIT_OPTIONS = 'RQS0'          # range, result, quiet, no draw
FIT_WEIGHT_OPTION = 'W'       # 'W': unit weights; '': use bin errors
EFFECTIVE_FIT_OPTIONS = FIT_OPTIONS + FIT_WEIGHT_OPTION
FIT_WEIGHT_TAG = 'weights1' if FIT_WEIGHT_OPTION == 'W' else 'weightsStd'
OUTPUT_CONFIGURATION_TAG = (
    f'fullFit_{FULL_FIT_FUNCTION}_fbFit_{FB_FIT_FUNCTION}'
    f'_{FIT_WEIGHT_TAG}_systCombMax'
)
SHOW_FIT_RESULTS = True

DRAW_GRID = True
SAVE_PNG = False
FULL_RATIO_RANGE = (0.85, 1.15)
FB_RANGE = (0.75, 1.30)
FB_DOUBLE_RATIO_RANGE = (0.85, 1.15)
SYSTEMATIC_Y_RANGE = None
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_POINTING_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'systematics_pointing_resolution',
))

if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError('GENERATOR must be embedding or pythia')
if DIRECTION not in ('combined', 'Pbgoing', 'pgoing'):
    raise ValueError('DIRECTION must be combined, Pbgoing, or pgoing')
if SYSTEMATIC_EXTRACTION not in ('fit', 'bin_by_bin'):
    raise ValueError("SYSTEMATIC_EXTRACTION must be 'fit' or 'bin_by_bin'")
if FORWARD_BACKWARD_RATIO_OPTION != '':
    raise ValueError('F/B construction must use independent errors')
if FIT_WEIGHT_OPTION not in ('', 'W'):
    raise ValueError("FIT_WEIGHT_OPTION must be empty or 'W'")
for option_name, option in (
    ('FULL_COMPARISON_RATIO_OPTION', FULL_COMPARISON_RATIO_OPTION),
    ('FB_COMPARISON_RATIO_OPTION', FB_COMPARISON_RATIO_OPTION),
):
    if option not in ('', 'B'):
        raise ValueError(f'{option_name} must be empty or B')
if not 0 <= ETA_CUT_INDEX < len(ETA_CUTS):
    raise IndexError(f'Invalid ETA_CUT_INDEX: {ETA_CUT_INDEX}')

ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
if DIRECTION == 'combined':
    INPUT_FILE = resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
else:
    INPUT_FILE = resolve_direction_file(
        BASE_DIR, GENERATOR, DIRECTION, FILE_STEM,
    )
if not INPUT_FILE.exists():
    raise FileNotFoundError(INPUT_FILE)

SOURCE_KEY = f'hRecoDijetPtEtaCMRefEtaCM_{ETA_CUT_INDEX}'
STYLE = replace(
    DEFAULT_PLOT_STYLE,
    annotation_text_size=0.026,
    annotation_line_spacing=0.039,
    legend_text_size=0.028,
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_FILE

PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/embedding/embedding_jetId.root')

In [13]:
def build_eta_projections(source, ptave_range, name_prefix):
    if not source.InheritsFrom('TH3'):
        raise TypeError(f'{SOURCE_KEY} is not a TH3 histogram')

    projections = {
        'Reco': project_histogram(
            source, {'axis': 'y', 'x_range': ptave_range},
            name=f'{name_prefix}_reco_raw',
        ),
        'Ref': project_histogram(
            source, {'axis': 'z', 'x_range': ptave_range},
            name=f'{name_prefix}_ref_raw',
        ),
    }
    if REBIN_ETA > 1:
        for histogram in projections.values():
            histogram.Rebin(REBIN_ETA)
    return projections


def build_systematic_band(systematic):
    graph = ROOT.TGraphAsymmErrors(systematic.GetNbinsX())
    for bin_index in range(1, systematic.GetNbinsX() + 1):
        uncertainty = systematic.GetBinContent(bin_index)
        half_width = systematic.GetBinWidth(bin_index) / 2.0
        graph.SetPoint(
            bin_index - 1, systematic.GetBinCenter(bin_index), 1.0,
        )
        graph.SetPointError(
            bin_index - 1, half_width, half_width,
            uncertainty, uncertainty,
        )
    graph.SetFillColorAlpha(COLORS[3], 0.30)
    graph.SetLineColor(COLORS[3])
    return graph


def draw_ratio_with_systematic_band(
    ratios, systematic, fit_functions, *, x_range, y_range,
    x_title, y_title, annotations, output, canvas_name,
):
    canvas = draw_overlay(
        ratios, title='', x_title=x_title, y_title=y_title,
        x_range=x_range, y_range=y_range, reference_y=1.0,
        annotations=annotations, grid=DRAW_GRID,
        overlay_functions=fit_functions,
        style_indices={next(iter(ratios)): 0}, style=STYLE,
        output=None, canvas_name=canvas_name,
    )
    graph = build_systematic_band(systematic)
    graph.Draw('E2 SAME')
    for histogram in ratios.values():
        histogram.Draw('E1 SAME')
    for function in fit_functions.values():
        function.Draw('SAME')

    legend = canvas._overlay_objects[0]
    legend.AddEntry(graph, 'Pointing syst. uncrt.', 'f')
    canvas._overlay_objects.append(graph)
    canvas.Modified()
    canvas.Update()
    save_canvas(canvas, output, save_png=SAVE_PNG)
    return canvas

In [14]:
source = load_histogram(str(INPUT_FILE), SOURCE_KEY)
pointing_results = {}
eta_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_range = (0.0, ETA_CUT + 0.1)

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    tag = (
        f'{GENERATOR}_{DIRECTION}_pointing_etaCM_{round(10 * ETA_CUT):g}'
        f'_ptave_{ptave_tag}'
    )
    output_prefix = f'{GENERATOR}_{DIRECTION}_pointing'
    output_suffix = (
        f'{OUTPUT_CONFIGURATION_TAG}_etaCM_{round(10 * ETA_CUT):g}'
        f'_ptave_{ptave_tag}'
    )

    raw_projections = build_eta_projections(source, ptave_range, f'h_{tag}')
    eta_shapes = {
        label: normalize_histogram(histogram, 'integral')
        for label, histogram in raw_projections.items()
    }
    fb_ratios = {
        label: forward_backward_from_full(
            histogram, name=f'h_{tag}_{label.lower()}_fb',
        )
        for label, histogram in raw_projections.items()
    }

    eta_ratios = {
        'Reco / Ref': ratio_to_nominal(
            eta_shapes['Reco'], eta_shapes['Ref'],
            name=f'h_{tag}_reco_ref', option=FULL_COMPARISON_RATIO_OPTION,
        )
    }
    fb_comparisons = {
        '(F/B) Reco / Ref': ratio_to_nominal(
            fb_ratios['Reco'], fb_ratios['Ref'],
            name=f'h_{tag}_fb_reco_ref', option=FB_COMPARISON_RATIO_OPTION,
        )
    }

    eta_fits, eta_fit_summary = fit_histogram_variations(
        eta_ratios, formula=FULL_FIT_FUNCTION,
        fit_range=(-ETA_CUT, ETA_CUT), name_prefix=f'f_{tag}_full',
        fit_options=EFFECTIVE_FIT_OPTIONS,
        initial_values=FULL_FIT_INITIAL_VALUES,
    )
    fb_fits, fb_fit_summary = fit_histogram_variations(
        fb_comparisons, formula=FB_FIT_FUNCTION,
        fit_range=(0.0, ETA_CUT), name_prefix=f'f_{tag}_fb',
        fit_options=EFFECTIVE_FIT_OPTIONS,
        initial_values=FB_FIT_INITIAL_VALUES,
    )

    use_fit = SYSTEMATIC_EXTRACTION == 'fit'
    eta_systematic_unsmoothed = calculate_one_sided_systematic(
        eta_ratios['Reco / Ref'], name=f'h_{tag}_full_syst',
        variation_function=eta_fits['Reco / Ref'] if use_fit else None,
        evaluation_range=(-ETA_CUT, ETA_CUT),
    )
    fb_systematic_unsmoothed = calculate_one_sided_systematic(
        fb_comparisons['(F/B) Reco / Ref'], name=f'h_{tag}_fb_syst',
        variation_function=(
            fb_fits['(F/B) Reco / Ref'] if use_fit else None
        ),
        evaluation_range=(0.0, ETA_CUT),
    )

    eta_systematic = (
        smooth_systematic_running_max(
            eta_systematic_unsmoothed,
            name=f'{eta_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(-ETA_CUT, ETA_CUT),
            smoothing_origin=FULL_SMOOTHING_ORIGIN,
        )
        if APPLY_SYSTEMATIC_SMOOTHING else eta_systematic_unsmoothed
    )
    fb_systematic = (
        smooth_systematic_running_max(
            fb_systematic_unsmoothed,
            name=f'{fb_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(0.0, ETA_CUT),
        )
        if APPLY_SYSTEMATIC_SMOOTHING else fb_systematic_unsmoothed
    )

    eta_percent = eta_systematic.Clone(f'{eta_systematic.GetName()}_percent')
    eta_percent.SetDirectory(0)
    eta_percent.Scale(100.0)
    fb_percent = fb_systematic.Clone(f'{fb_systematic.GetName()}_percent')
    fb_percent.SetDirectory(0)
    fb_percent.Scale(100.0)

    annotations = (
        GENERATOR.capitalize(), DIRECTION,
        f'{low:g} < p_{{T}}^{{ave,reco}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    common_plot_options = dict(
        title='', annotations=annotations, grid=DRAW_GRID,
        style=STYLE, save_png=SAVE_PNG,
    )
    canvases = {}
    canvases['eta_overlay'] = draw_overlay(
        eta_shapes, x_title='#eta_{CM}^{dijet}',
        y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_range,
        style_indices={'Reco': 0, 'Ref': 1},
        output=OUTPUT_DIR / f'{output_prefix}_full_overlay_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_overlay', **common_plot_options,
    )
    canvases['eta_ratio'] = draw_overlay(
        eta_ratios, x_title='#eta_{CM}^{dijet}', y_title='Reco / Ref',
        x_range=eta_range, y_range=FULL_RATIO_RANGE, reference_y=1.0,
        overlay_functions=eta_fits,
        overlay_text=(
            format_fit_summary_lines(eta_fit_summary)
            if SHOW_FIT_RESULTS else None
        ),
        style_indices={'Reco / Ref': 0},
        output=OUTPUT_DIR / f'{output_prefix}_full_ratio_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_ratio', **common_plot_options,
    )
    canvases['fb_overlay'] = draw_overlay(
        fb_ratios, x_title='|#eta_{CM}^{dijet}|',
        y_title='Forward / Backward', x_range=fb_range, y_range=FB_RANGE,
        reference_y=1.0, style_indices={'Reco': 0, 'Ref': 1},
        output=OUTPUT_DIR / f'{output_prefix}_fb_overlay_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_overlay', **common_plot_options,
    )
    canvases['fb_ratio'] = draw_overlay(
        fb_comparisons, x_title='|#eta_{CM}^{dijet}|',
        y_title='(F/B)_{Reco} / (F/B)_{Ref}', x_range=fb_range,
        y_range=FB_DOUBLE_RATIO_RANGE, reference_y=1.0,
        overlay_functions=fb_fits,
        overlay_text=(
            format_fit_summary_lines(fb_fit_summary)
            if SHOW_FIT_RESULTS else None
        ),
        style_indices={'(F/B) Reco / Ref': 0},
        output=OUTPUT_DIR / f'{output_prefix}_fb_ratio_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_ratio', **common_plot_options,
    )

    suffix = 'smoothed' if APPLY_SYSTEMATIC_SMOOTHING else 'nonsmoothed'
    eta_systematic_csv = write_systematic_csv(
        eta_systematic,
        OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_relative_{output_suffix}.csv',
        evaluation_range=(-ETA_CUT, ETA_CUT),
    )
    fb_systematic_csv = write_systematic_csv(
        fb_systematic,
        OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_fb_relative_{output_suffix}.csv',
        evaluation_range=(0.0, ETA_CUT),
    )
    canvases['eta_syst'] = draw_overlay(
        {'Pointing resolution': eta_percent},
        x_title='#eta_{CM}^{dijet}', y_title='Pointing Rel. Syst. Uncrt. (%)',
        x_range=eta_range, y_range=SYSTEMATIC_Y_RANGE, show_legend=False,
        output=OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_syst', **common_plot_options,
    )
    canvases['fb_syst'] = draw_overlay(
        {'Pointing resolution': fb_percent},
        x_title='|#eta_{CM}^{dijet}|', y_title='Pointing Rel. Syst. Uncrt. (%)',
        x_range=fb_range, y_range=SYSTEMATIC_Y_RANGE, show_legend=False,
        output=OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_fb_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_syst', **common_plot_options,
    )
    canvases['eta_band'] = draw_ratio_with_systematic_band(
        eta_ratios, eta_systematic, eta_fits, x_range=eta_range,
        y_range=FULL_RATIO_RANGE, x_title='#eta_{CM}^{dijet}',
        y_title='Reco / Ref', annotations=annotations,
        output=OUTPUT_DIR / f'{output_prefix}_full_ratio_band_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_band',
    )
    canvases['fb_band'] = draw_ratio_with_systematic_band(
        fb_comparisons, fb_systematic, fb_fits, x_range=fb_range,
        y_range=FB_DOUBLE_RATIO_RANGE, x_title='|#eta_{CM}^{dijet}|',
        y_title='(F/B)_{Reco} / (F/B)_{Ref}', annotations=annotations,
        output=OUTPUT_DIR / f'{output_prefix}_fb_ratio_band_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_band',
    )

    pointing_results[ptave_range] = {
        'raw': raw_projections, 'eta_shapes': eta_shapes,
        'forward_backward': fb_ratios, 'eta_ratios': eta_ratios,
        'fb_ratios': fb_comparisons, 'eta_fits': eta_fits,
        'fb_fits': fb_fits, 'eta_fit_summary': eta_fit_summary,
        'fb_fit_summary': fb_fit_summary,
        'eta_systematic_unsmoothed': eta_systematic_unsmoothed,
        'fb_systematic_unsmoothed': fb_systematic_unsmoothed,
        'eta_systematic': eta_systematic,
        'fb_systematic': fb_systematic,
        'eta_systematic_csv': eta_systematic_csv,
        'fb_systematic_csv': fb_systematic_csv,
        'canvases': canvases,
    }
    print(ptave_range, eta_fit_summary, fb_fit_summary)
    for canvas in canvases.values():
        display(canvas)

(60, 100) {'Reco / Ref': {'formula': 'pol4', 'range': (-1.9, 1.9), 'fit_bin_center_range': (-1.8999999999999997, 1.900000000000001), 'initial_parameters': (1.0, 0.0, 0.0, 0.0, 0.0), 'parameters': (1.002445672990532, -0.001006990905732105, -0.0001645853174758897, -0.0003519646171155753, -0.002005048309894675), 'parameter_errors': (0.0002075096476530071, 0.0004912431587478163, 0.0008939908319119819, 0.0002956730605859506, 0.00038529627151569657), 'covariance': ((4.306025386907516e-08, 3.4491183250446536e-08, -1.175997580549206e-07, -1.8214488442757482e-08, 4.2523715956050305e-08), (3.4491183250446536e-08, 2.4131984101653223e-07, -9.25741817170866e-08, -1.2144621865839697e-07, 3.185253990092177e-08), (-1.175997580549206e-07, -9.25741817170866e-08, 7.992196075426775e-07, 5.2335926494444295e-08, -3.2893357606122436e-07), (-1.8214488442757482e-08, -1.2144621865839697e-07, 5.2335926494444295e-08, 8.74225587562632e-08, -1.3557758209729975e-08), (4.2523715956050305e-08, 3.185253990092177e-08, -

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_5281912515963228356
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_60_100_full_overlay.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_8504981925459189252
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_60_100_full_ratio.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_8639978240289632528
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_60_100_fb_overlay.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_182974558415

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_7160804568765364761
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_120_180_full_overlay.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_619109576573409671


(120, 180) {'Reco / Ref': {'formula': 'pol4', 'range': (-1.9, 1.9), 'fit_bin_center_range': (-1.8999999999999997, 1.900000000000001), 'initial_parameters': (1.0, 0.0, 0.0, 0.0, 0.0), 'parameters': (1.001020749325169, -0.0008496063035473368, 0.00031540837002113226, 0.00041370879253336895, -0.0008488708251216645), 'parameter_errors': (7.048434628557197e-05, 0.00014253737523109032, 0.00023675967868253233, 0.00013300977094569206, 0.00013461337932585266), 'covariance': ((4.968043071304422e-09, -4.383305979759009e-09, -1.0835234772405583e-08, 2.7450349806376266e-09, 4.192706722862786e-09), (-4.383305979759009e-09, 2.031690333776864e-08, -1.6681250969766958e-09, -1.5780979141205012e-08, 2.288101468128874e-09), (-1.0835234772405583e-08, -1.6681250969766958e-09, 5.6055145449855954e-08, 2.698135725577192e-09, -2.8309180438867457e-08), (2.7450349806376266e-09, -1.5780979141205012e-08, 2.698135725577192e-09, 1.7691599167025472e-08, -1.2658969685880297e-09), (4.192706722862786e-09, 2.28810146812887

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_120_180_full_ratio.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_3489234714472718502
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_120_180_fb_overlay.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_4088150565277420012
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_120_180_fb_ratio.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_8005766403254367639
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/outpu

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_8006929971158819469
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_200_300_full_overlay.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_7157973845549971787
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_200_300_full_ratio.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_7698041526081786870
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_200_300_fb_overlay.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_706073776

(200, 300) {'Reco / Ref': {'formula': 'pol4', 'range': (-1.9, 1.9), 'fit_bin_center_range': (-1.8999999999999997, 1.900000000000001), 'initial_parameters': (1.0, 0.0, 0.0, 0.0, 0.0), 'parameters': (1.00012348413115, -0.0012328884621129112, -0.0007939649086060035, 0.00017196654036936632, -7.562742405033455e-05), 'parameter_errors': (0.00010971122026740739, 0.0002580924133648492, 0.0003442766812709891, 0.00021970537039917216, 0.0002016246524152624), 'covariance': ((1.203655185256358e-08, 3.672998001474806e-10, -2.271026890030628e-08, -6.249835889724488e-11, 8.339791252097804e-09), (3.672998001474806e-10, 6.66116938364922e-08, 7.506196918525639e-09, -5.023500253293611e-08, -1.3030373191039003e-08), (-2.271026890030628e-08, 7.506196918525639e-09, 1.185264332669662e-07, -1.4354010904027282e-08, -6.228632469703498e-08), (-6.249835889724488e-11, -5.023500253293611e-08, -1.4354010904027282e-08, 4.827044978223743e-08, 1.9341248492674375e-08), (8.339791252097804e-09, -1.3030373191039003e-08, -6.

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_200_300_systematic_smoothed_fb.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_7821637800005221713
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_200_300_full_ratio_band.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_4587907204250360713
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_200_300_fb_ratio_band.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_8807381580031737825
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalys

(300, 500) {'Reco / Ref': {'formula': 'pol4', 'range': (-1.9, 1.9), 'fit_bin_center_range': (-1.8999999999999997, 1.900000000000001), 'initial_parameters': (1.0, 0.0, 0.0, 0.0, 0.0), 'parameters': (0.9999656737021555, 0.000539354651900492, -0.0007917517714144014, -0.00017747347817183667, 0.0003319519368850302), 'parameter_errors': (4.641156879515633e-05, 0.00013823781233235276, 0.00017222655621444875, 0.00012933108178045448, 0.00011802071867302079), 'covariance': ((2.1540337180275286e-09, -1.0005607961983583e-09, -4.292640521582584e-09, 7.673027546366504e-10, 1.8482915280229908e-09), (-1.0005607961983583e-09, 1.9109692758434778e-08, 2.767240251090042e-09, -1.6046093138267905e-08, -4.001993693994887e-09), (-4.292640521582584e-09, 2.767240251090042e-09, 2.966198666548868e-08, -4.026252841408056e-09, -1.8221672210648558e-08), (7.673027546366504e-10, -1.6046093138267905e-08, -4.026252841408056e-09, 1.6726528714502605e-08, 6.259257992969722e-09), (1.8482915280229908e-09, -4.001993693994887e

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_300_500_fb_overlay.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_5760919033155388416
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_300_500_fb_ratio.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_5490615378928908242
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/systematics_pointing_resolution/embedding_combined_pointing_etaCM_19_ptave_300_500_systematic_smoothed_full.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_2823957592343198626
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_

In [15]:
for ptave_range, result in pointing_results.items():
    normalized_integrals = {
        label: histogram.Integral()
        for label, histogram in result['eta_shapes'].items()
    }
    assert all(
        abs(integral - 1.0) < 1e-9
        for integral in normalized_integrals.values()
    ), normalized_integrals
    for histogram in result['forward_backward'].values():
        assert hasattr(histogram, '_forward_backward_inputs')
        assert not histogram.GetDirectory()

    raw_yields = {
        label: histogram.Integral()
        for label, histogram in result['raw'].items()
    }
    print(
        ptave_range, 'normalized integrals', normalized_integrals,
        'raw yields', raw_yields,
    )

(60, 100) normalized integrals {'Reco': 0.9999999999999999, 'Ref': 0.9999999999999998} raw yields {'Reco': 0.0024567719346745153, 'Ref': 0.0024567719346745153}
(120, 180) normalized integrals {'Reco': 1.0, 'Ref': 0.9999999999999999} raw yields {'Reco': 0.00015468439487003886, 'Ref': 0.00015468439487003891}
(200, 300) normalized integrals {'Reco': 1.0000000000000002, 'Ref': 0.9999999999999998} raw yields {'Reco': 1.679598092024606e-05, 'Ref': 1.679598092024606e-05}
(300, 500) normalized integrals {'Reco': 1.0, 'Ref': 1.0000000000000002} raw yields {'Reco': 2.6530641752548182e-06, 'Ref': 2.6530641752548182e-06}
